# Instalar as dependências para o R:

In [ ]:
# 1. Instalar dependências do sistema Linux necessárias para manipulação de dados e FTP
system("apt-get update && apt-get install -y libssl-dev libcurl4-openssl-dev libxml2-dev r-base-dev build-essential")

# 2. Instalar pacotes do R

options(repos = c(CRAN = "https://cloud.r-project.org"))

install.packages("remotes")
install.packages("rlang")
install.packages("tidyverse") # Inclui dplyr, ggplot2, etc.

# 3. Instalar o pacote microdatasus direto do repositório da Fiocruz (vai utilizar o CRAN setado anteriormente)
remotes::install_github("rfsaldanha/microdatasus", type = "source")

# 4. Carregar as bibliotecas
library("remotes")
library(microdatasus)
library(dplyr)

# As bibliotecas instaladas ficam em: /opt/conda/lib/R/library

Carregar/Verificar as bibliotecas:

In [8]:
library("remotes")
library(microdatasus)
library(dplyr)
#.libPaths()

# Baixando os dados referentes às bases SINAN para os anos de 2010 à 2025, com excessão de 2020 e 2021 (ausentes no SINAN).:

In [ ]:
anos_antigos <- "10 11 12 13 14 15 16 17 18 19"
anos_recentes <- "22 23 24 25"

system(paste("for ano in", anos_antigos, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/FINAIS/MENTBR$ano.dbc; done"))
system(paste("for ano in", anos_recentes, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR$ano.dbc; done"))

#system("wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR24.dbc")

library(read.dbc)
dados <- read.dbc("~/work/MENTBR25.dbc")

head(dados)

Lendo e concatenando todos pacotes baixados referentes a cada ano:

In [ ]:
#library(read.dbc)
#library(dplyr)

# 1. Definir os anos que queremos (2010 a 2025, removendo 2020 e 2021)
anos <- 2010:2025
anos_filtrados <- anos[!anos %in% c(2020, 2021)]

# 2. Criar uma lista vazia para armazenar os dataframes temporariamente
lista_dados <- list()

# 3. O Laço de Repetição
for (ano in anos_filtrados) {

  # Extrai os dois últimos dígitos do ano (ex: 2015 vira "15")
  sufixo <- substr(as.character(ano), 3, 4)

  # Monta o caminho do arquivo dinamicamente
  caminho_arquivo <- paste0("~/work/MENTBR", sufixo, ".dbc")

  # Cria um nome de identificação para a lista (ex: "ano_15")
  nome_lista <- paste0("ano_", sufixo)

  # Mensagem no console para você acompanhar o progresso da leitura
  cat("Lendo arquivo:", caminho_arquivo, "\n")

  # Lê o arquivo e armazena na lista se ele existir
  if (file.exists(caminho_arquivo)) {
    lista_dados[[nome_lista]] <- read.dbc(caminho_arquivo)
  } else {
    warning(paste("Arquivo não encontrado:", caminho_arquivo))
  }
}

# 4. Junta todos os anos em um único grande dataframe
dados_completos <- bind_rows(lista_dados, .id = "FONTE_ANO")

# Remove a lista temporária para liberar memória RAM
rm(lista_dados)

# Salvando na memória secundária
saveRDS(dados_completos, file = "work/dados_completos_sinan.rds")

# dados <- readRDS("work/dados_completos_sinan.rds")

In [12]:
dados <- readRDS("~/work/dados_completos_sinan.rds")

In [13]:
head(dados)

,FONTE_ANO,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,⋯,ANO_NASC,NU_LOTE_I,DT_TRANSUS,DT_TRANSDM,DT_TRANSSM,DT_TRANSRM,DT_TRANSRS,DT_TRANSSE,NU_LOTE_V,NU_LOTE_H
,<chr>,<fct>,<fct>,<date>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<fct>,<fct>,<date>,<date>,<date>,<date>,<date>,<date>,<fct>,<fct>
1,ano_10,2,F99,2010-01-27,201004,2010,28,280030,2056,5841399,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,ano_10,2,F99,2010-01-29,201004,2010,42,420910,1565,2436418,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,ano_10,2,F99,2010-01-05,201001,2010,35,355030,1331,2786664,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,ano_10,2,F99,2010-01-26,201004,2010,29,293330,1398,2550202,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
5,ano_10,2,F99,2010-01-15,201002,2010,29,292740,1380,2557894,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,ano_10,2,F99,2010-01-15,201002,2010,29,291480,1386,3432890,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
